# Figure 5E — Lymphocyte marker gene expression in MHC II high vs low tumors

**Goal:** Compare expression of effector and functional marker genes across
lymphocyte cell types between MHC II high and low LUAD tumors. Percent of
cells expressing each gene is computed per sample, then compared between
MHC II classifications using Mann-Whitney U with BH FDR correction applied
across all gene × cell type combinations.

**Input:** `outputs/processed/luad_mhc2_classified.h5ad`

**Output:** Figure 5E saved to `outputs/figures/figure5/`

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import yaml

sns.set(font_scale=2)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

palette = {'MHC class II High': '#FF8811FF', 'MHC class II Low': '#462255FF'}
group_order = ['MHC class II High', 'MHC class II Low']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_out   = repo_root / cfg['outputs']['figures'] / 'figure5'
table_out = repo_root / cfg['outputs']['tables']  / 'figure5'

fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load Data

In [ ]:
tumordata = sc.read_h5ad(repo_root / 'outputs/processed/luad_mhc2_classified.h5ad')

# restrict to classified samples
tumordata = tumordata[
    tumordata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()

print(f"Cells: {tumordata.n_obs:,}")
print(tumordata.obs['MHC2_clustering'].value_counts())

## Gene and cell type selection

Effector and functional marker genes selected for lymphocyte phenotyping.
Percent of cells expressing each gene (expression > 0) is used as the
metric — this is robust to zero-inflation which is common for effector
molecules in scRNA-seq data.

In [ ]:
genes_of_interest = ['EOMES', 'LAG3', 'GZMA', 'GZMB', 'GZMK', 'GZMH']
cell_types        = ['T cell CD4', 'T cell CD8', 'T cell regulatory', 'NK cell']

# map gene symbols to Ensembl IDs
genes_dict = {}
for g in genes_of_interest:
    match = tumordata.var[tumordata.var['feature_name'] == g]
    if len(match) > 0:
        genes_dict[g] = list(match.index)
    else:
        print(f'Warning: {g} not found in var')

ens_to_name = {ens: g for g, ensembls in genes_dict.items() for ens in ensembls}
print(f"Genes mapped: {list(genes_dict.keys())}")

## Compute percent expressing per sample and run statistics

Mann-Whitney U tests are run for each gene × cell type combination.
FDR correction is applied across all combinations simultaneously
using Benjamini-Hochberg.

In [ ]:
stats_records = []
pct_data = {}

for cell_type in cell_types:
    subset = tumordata[tumordata.obs['cell_type_major'] == cell_type]
    pct_data[cell_type] = {}

    for gene, gene_ens_list in genes_dict.items():
        gene_ens = gene_ens_list[0]

        expr_bin = (subset.to_df()[gene_ens] > 0).astype(int)
        expr_bin = expr_bin.to_frame(name='expr')
        expr_bin['sample']  = subset.obs['sample'].values
        expr_bin['cluster'] = subset.obs['MHC2_clustering'].values

        pct_df = (
            expr_bin
            .groupby(['sample', 'cluster'], observed=True)['expr']
            .mean()
            .reset_index()
        )
        pct_df['expr'] = pct_df['expr'] * 100
        pct_data[cell_type][gene] = pct_df

        g1 = pct_df.loc[pct_df['cluster'] == 'MHC class II High', 'expr']
        g2 = pct_df.loc[pct_df['cluster'] == 'MHC class II Low',  'expr']

        if len(g1) > 0 and len(g2) > 0:
            _, pval = mannwhitneyu(g1, g2, alternative='two-sided')
        else:
            pval = np.nan

        stats_records.append({
            'cell_type': cell_type,
            'gene':      gene,
            'p_value':   pval,
            'mean_high': g1.mean(),
            'mean_low':  g2.mean(),
        })

stats_df = pd.DataFrame(stats_records)

# FDR correction across all gene × cell type combinations
_, fdr, _, _ = multipletests(stats_df['p_value'].fillna(1), method='fdr_bh')
stats_df['FDR_p'] = fdr
stats_df['Significant'] = fdr < 0.05
stats_df['sig_label'] = stats_df['FDR_p'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
)

stats_df.to_csv(table_out / 'fig5e_lymphocyte_marker_stats.csv', index=False)
print(stats_df[['cell_type', 'gene', 'p_value', 'FDR_p', 'sig_label']].to_string(index=False))

## Figure 5E — Percent expressing per gene and cell type

In [ ]:
nrows = len(cell_types)
ncols = len(genes_of_interest)

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(4 * ncols, 4 * nrows),
                         sharey=False)
axes = np.atleast_2d(axes)

for r, cell_type in enumerate(cell_types):
    for c, gene in enumerate(genes_of_interest):
        ax = axes[r, c]
        plot_df = pct_data[cell_type][gene].copy()

        sns.boxplot(
            data=plot_df, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, fill=False, showfliers=False, legend=False,
        )
        sns.stripplot(
            data=plot_df, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, edgecolor='k', linewidth=1,
            size=4, alpha=0.7, legend=False,
        )

        if r == 0:
            ax.set_title(gene, fontsize=22, pad=15)  # increase pad from default
            
        if c == 0:
            ax.set_ylabel(f'{cell_type}\n% cells expressing', fontsize=18)
        else:
            ax.set_ylabel('')

        ax.set_xlabel('')
        ax.set_xticklabels([])
        ax.spines[['top', 'right']].set_visible(False)

        # FDR-corrected significance label
        sig = stats_df.loc[
            (stats_df['cell_type'] == cell_type) &
            (stats_df['gene'] == gene), 'sig_label'
        ].values[0]
        if sig:
            ax.text(0.5, 0.72, sig, ha='center', va='bottom',
                    transform=ax.transAxes, fontsize=48)

# shared legend
handles = [
    plt.Line2D([0], [0], color=palette[g], marker='o', linestyle='',
               markersize=8, markeredgecolor='k', markeredgewidth=1, label=g)
    for g in group_order
]
fig.legend(handles=handles, loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=16)

fig.suptitle('Percent of cells expressing selected genes across cell types by MHC2 cluster',
             fontsize=16, y=1.05)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(fig_out / 'fig5e_lymphocyte_markers.pdf', bbox_inches='tight')
plt.show()

## CD4+ T cell subset analysis

Subset to CD4+ T cells for deeper characterization of the cytotoxic
CD4+ phenotype. New UMAP coordinates are computed on the CD4+ subset
alone using Harmony batch correction by study, since the full atlas
UMAP embeds all cell types together and loses resolution within T cell
subsets.

In [ ]:
%%time 

import harmonypy

cd4 = tumordata[tumordata.obs['cell_type_major'] == 'T cell CD4'].copy()
print(f"CD4+ T cells: {cd4.n_obs:,}")

cd4_path = repo_root / 'outputs/processed/cd4_tcell_mhc2_umap.h5ad'

if cd4_path.exists():
    cd4 = sc.read_h5ad(cd4_path)
    print("Loaded from checkpoint")
else:
    # HVG
    # use seurat flavor which works on normalized X
    sc.pp.highly_variable_genes(cd4, n_top_genes=2000, flavor='seurat', batch_key='study')

    # PCA
    sc.pp.pca(cd4, n_comps=50, use_highly_variable=True)
    # Harmony
    ho = harmonypy.run_harmony(cd4.obsm['X_pca'], cd4.obs, 'study',
                                max_iter_harmony=20)
    cd4.obsm['X_pca_harmony'] = ho.Z_corr.T if ho.Z_corr.shape[1] == cd4.n_obs else ho.Z_corr
    # neighbors + UMAP
    sc.pp.neighbors(cd4, use_rep='X_pca_harmony', n_neighbors=30)
    sc.tl.umap(cd4)
    cd4.write_h5ad(cd4_path)
    print("Computed and saved checkpoint")

## UMAPs — CD4+ T cell subset

UMAPs colored by MHC II classification and key marker genes.
Cytotoxic markers (GZMK, GZMH, GZMA, EOMES, PRF1) identify
cytotoxic CD4+ subpopulation. Exhaustion markers (PDCD1, HAVCR2,
CTLA4, LAG3, TIGIT) characterize immune checkpoint expression.

In [ ]:
umap_genes = ['GZMK', 'GZMH', 'GZMA', 'EOMES', 'PRF1',
              'PDCD1', 'HAVCR2', 'CTLA4', 'LAG3', 'TIGIT', 'IFNG']

umap_genes_ens = {}
for g in umap_genes:
    match = cd4.var[cd4.var['feature_name'] == g]
    if len(match) > 0:
        umap_genes_ens[g] = match.index[0]
    else:
        print(f'Warning: {g} not found')

print(umap_genes_ens)

In [ ]:
sc.settings.figdir = str(fig_out)

sc.pl.umap(
    cd4,
    color='MHC2_clustering',
    palette={'MHC class II High': palette['MHC class II High'],
             'MHC class II Low':  palette['MHC class II Low']},
    title='CD4+ T cells — MHC II classification',
    frameon=False,
    save='_fig5e_cd4_mhc2_clustering.pdf',
)

In [ ]:
cytotoxic_genes = ['GZMK', 'GZMH', 'GZMA', 'EOMES', 'PRF1']
cytotoxic_ens   = [umap_genes_ens[g] for g in cytotoxic_genes if g in umap_genes_ens]

sc.pl.umap(
    cd4,
    color=cytotoxic_ens,
    gene_symbols='feature_name',
    ncols=5,
    frameon=False,
    cmap='Reds',
    title=cytotoxic_genes,
    size = 2,
    vmin=0,
    vmax='p99',  # clip at 99th percentile to avoid outlier domination
    use_raw = False,
    save='_fig5e_cd4_cytotoxic_markers.pdf',
)

In [ ]:
exhaustion_genes = ['PDCD1', 'HAVCR2', 'CTLA4', 'LAG3', 'TIGIT']
exhaustion_ens   = [umap_genes_ens[g] for g in exhaustion_genes if g in umap_genes_ens]

sc.pl.umap(
    cd4,
    color=exhaustion_ens,
    gene_symbols='feature_name',
    ncols=5,
    frameon=False,
    cmap='Purples',
    size = 2,
    vmin=0,
    vmax='p99',  # clip at 99th percentile to avoid outlier domination
    title=exhaustion_genes,
    use_raw = False,
    save='_fig5e_cd4_exhaustion_markers.pdf',
)

In [ ]:
%%time 

sc.tl.leiden(cd4, resolution=0.5)
sc.pl.umap(cd4, color='leiden', legend_loc='on data', frameon=False)

In [ ]:
%%time 

sc.tl.embedding_density(cd4, groupby='MHC2_clustering')
sc.pl.embedding_density(cd4, groupby='MHC2_clustering', ncols=2)

## CD4 T cells from either patient groups tend to appear in different regions of the UMAP. Let's look at the differentially expressed genes in the various leiden groups

In [ ]:
sc.tl.rank_genes_groups(cd4, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(cd4, n_genes=20, sharey=False, gene_symbols = 'feature_name')

In [ ]:
# proportion of cells in each cluster per MHC2 group
cluster_props = (
    cd4.obs.groupby(['MHC2_clustering', 'leiden'], observed=True)
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='pct')
)
print(cluster_props[cluster_props['leiden'] == '2'])

In [ ]:
# proportion of cytotoxic CD4+ cluster per patient
cell_cluster_props = (
    cd4.obs.groupby(['sample', 'MHC2_clustering', 'leiden'], observed=True)
    .size()
    .reset_index(name='count')
)
total_per_sample = (
    cd4.obs.groupby(['sample', 'MHC2_clustering'], observed=True)
    .size()
    .reset_index(name='total')
)
cell_cluster_props = cell_cluster_props.merge(total_per_sample, on=['sample', 'MHC2_clustering'])
cell_cluster_props['pct'] = cell_cluster_props['count'] / cell_cluster_props['total'] * 100

# filter to cluster 2
cluster2_props = cell_cluster_props[
    (cell_cluster_props['leiden'] == '2') &
    (cell_cluster_props['MHC2_clustering'].isin(group_order))
].copy()

# test
g1 = cluster2_props.loc[cluster2_props['MHC2_clustering'] == 'MHC class II High', 'pct']
g2 = cluster2_props.loc[cluster2_props['MHC2_clustering'] == 'MHC class II Low',  'pct']
_, p = mannwhitneyu(g1, g2, alternative='two-sided')
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
print(f'Cluster 2 enrichment: p={p:.3e} ({sig})  mean_high={g1.mean():.1f}%  mean_low={g2.mean():.1f}%')

# plot
fig, ax = plt.subplots(figsize=(4, 5))
sns.boxplot(
    data=cluster2_props, x='MHC2_clustering', y='pct', hue='MHC2_clustering',
    order=group_order, palette=palette,
    ax=ax, fill=False, showfliers=False, legend=False,
)
sns.stripplot(
    data=cluster2_props, x='MHC2_clustering', y='pct', hue='MHC2_clustering',
    order=group_order, palette=palette,
    ax=ax, edgecolor='k', linewidth=1, size=5, alpha=0.7, legend=False,
)
ax.text(0.5, 0.88, f'{sig}\np={p:.2e}', ha='center', va='bottom',
        transform=ax.transAxes, fontsize=12)
ax.set_title('Cytotoxic CD4+ T cells\n(CCL5+GZMA+HLA-DRA)', fontsize=12)
ax.set_ylabel('% of CD4+ T cells')
ax.set_xlabel('')
ax.set_xticklabels([])
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_out / 'fig5e_cytotoxic_cd4_cluster_enrichment.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 1. Is cluster 2 enriched specifically or is it all clusters shifted?
print(
    cd4.obs.groupby(['MHC2_clustering', 'leiden'], observed=True)
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='pct')
    .pivot(index='leiden', columns='MHC2_clustering', values='pct')
    .round(1)
)

# 2. What fraction of cluster 2 cells come from MHC II high vs low patients?
print(cd4.obs[cd4.obs['leiden'] == '2']['MHC2_clustering'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
prop_df = (
    cd4.obs.groupby(['MHC2_clustering', 'leiden'], observed=True)
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='pct')
)

# assign cluster labels
cluster_labels = {
    '0': 'Stressed/mito',
    '1': 'B cell contam.',
    '2': 'Cytotoxic CD4+',
    '3': 'Ribosomal/prolif',
    '4': 'Heat shock',
    '5': 'IFN-stimulated',
    '6': 'NK-like CD4+',
}
prop_df['cluster_label'] = prop_df['leiden'].map(cluster_labels)

pivot = prop_df.pivot(index='MHC2_clustering', columns='cluster_label', values='pct')

fig, ax = plt.subplots(figsize=(7, 5))
pivot.plot(kind='bar', stacked=True, ax=ax,
           colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_xlabel('')
ax.set_ylabel('% of CD4+ T cells')
ax.set_xticklabels(['MHC II High', 'MHC II Low'], rotation=0)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_out / 'fig5e_cd4_cluster_proportions.pdf', bbox_inches='tight')
plt.show()

## EOMES and IFNG expression analysis

Mean expression per sample compared between MHC II high and low patients,
both across all CD4+ T cells and restricted to EOMES+ or IFNG+ cells only.
Wilcoxon signed-rank test on per-sample means.

In [ ]:
from scipy.stats import mannwhitneyu

target_genes = ['EOMES', 'IFNG']

for gene in target_genes:
    gene_ens = umap_genes_ens.get(gene)
    if gene_ens is None:
        print(f'{gene} not found')
        continue

    x = cd4[:, gene_ens].X
    x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

    df = pd.DataFrame({
        'sample':  cd4.obs['sample'].values,
        'cluster': cd4.obs['MHC2_clustering'].values,
        gene:      x,
    })

    # all cells mean
    mean_all = (
        df.groupby(['sample', 'cluster'], observed=True)[gene]
        .mean().reset_index()
    )

    # mean among positive cells only
    mean_pos = (
        df[df[gene] > 0]
        .groupby(['sample', 'cluster'], observed=True)[gene]
        .mean().reset_index()
    )

    for label, data in [('all cells', mean_all), (f'{gene}+ cells only', mean_pos)]:
        high = data.loc[data['cluster'] == 'MHC class II High', gene]
        low  = data.loc[data['cluster'] == 'MHC class II Low',  gene]
        if len(high) > 0 and len(low) > 0:
            _, p = mannwhitneyu(high, low, alternative='two-sided')
            print(f'{gene} ({label}): p={p:.3e}  mean_high={high.mean():.3f}  mean_low={low.mean():.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for col_i, gene in enumerate(target_genes):
    gene_ens = umap_genes_ens.get(gene)
    x = cd4[:, gene_ens].X
    x = x.toarray().ravel() if hasattr(x, 'toarray') else np.asarray(x).ravel()

    df = pd.DataFrame({
        'sample':  cd4.obs['sample'].values,
        'cluster': cd4.obs['MHC2_clustering'].values,
        gene:      x,
    })

    mean_all = (
        df.groupby(['sample', 'cluster'], observed=True)[gene]
        .mean().reset_index()
    )
    mean_pos = (
        df[df[gene] > 0]
        .groupby(['sample', 'cluster'], observed=True)[gene]
        .mean().reset_index()
    )

    for row_i, (label, data) in enumerate([
        ('Mean expression\n(all cells)', mean_all),
        (f'Mean expression\n({gene}+ cells only)', mean_pos),
    ]):
        ax = axes[row_i, col_i]
        sns.boxplot(
            data=data, x='cluster', y=gene, hue='cluster',
            order=group_order, palette=palette,
            ax=ax, fill=False, showfliers=False, legend=False,
        )
        sns.stripplot(
            data=data, x='cluster', y=gene, hue='cluster',
            order=group_order, palette=palette,
            ax=ax, edgecolor='k', linewidth=1,
            size=5, alpha=0.7, legend=False,
        )
        high = data.loc[data['cluster'] == 'MHC class II High', gene]
        low  = data.loc[data['cluster'] == 'MHC class II Low',  gene]
        _, p = mannwhitneyu(high, low, alternative='two-sided')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.text(0.5, 0.88, f'{sig}\np={p:.2e}', ha='center', va='bottom',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(f'{gene} — {label}', fontsize=12)
        ax.set_xlabel('')
        ax.set_xticklabels([])
        ax.set_ylabel('Mean expression')
        ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(fig_out / 'fig5e_eomes_ifng_expression.pdf', bbox_inches='tight')
plt.show()

## Exhaustion marker panel

Percent of cells expressing exhaustion markers (PDCD1, HAVCR2, CTLA4,
LAG3, TIGIT) across lymphocyte cell types. FDR correction applied across
all gene × cell type combinations. A separate panel from the cytotoxic
marker figure to avoid overloading Figure 5E.

In [ ]:
exhaustion_genes_list = ['PDCD1', 'HAVCR2', 'CTLA4', 'LAG3', 'TIGIT']

# map to ensembl
exhaust_dict = {}
for g in exhaustion_genes_list:
    match = tumordata.var[tumordata.var['feature_name'] == g]
    if len(match) > 0:
        exhaust_dict[g] = list(match.index)

# compute pct expressing + stats
exhaust_stats = []
exhaust_pct   = {}

for cell_type in cell_types:
    subset = tumordata[tumordata.obs['cell_type_major'] == cell_type]
    exhaust_pct[cell_type] = {}

    for gene, gene_ens_list in exhaust_dict.items():
        gene_ens = gene_ens_list[0]
        expr_bin = (subset.to_df()[gene_ens] > 0).astype(int)
        expr_bin = expr_bin.to_frame(name='expr')
        expr_bin['sample']  = subset.obs['sample'].values
        expr_bin['cluster'] = subset.obs['MHC2_clustering'].values

        pct_df = (
            expr_bin
            .groupby(['sample', 'cluster'], observed=True)['expr']
            .mean().reset_index()
        )
        pct_df['expr'] = pct_df['expr'] * 100
        exhaust_pct[cell_type][gene] = pct_df

        g1 = pct_df.loc[pct_df['cluster'] == 'MHC class II High', 'expr']
        g2 = pct_df.loc[pct_df['cluster'] == 'MHC class II Low',  'expr']
        if len(g1) > 0 and len(g2) > 0:
            _, p = mannwhitneyu(g1, g2, alternative='two-sided')
        else:
            p = np.nan
        exhaust_stats.append({'cell_type': cell_type, 'gene': gene, 'p_value': p})

exhaust_stats_df = pd.DataFrame(exhaust_stats)
_, fdr, _, _ = multipletests(exhaust_stats_df['p_value'].fillna(1), method='fdr_bh')
exhaust_stats_df['FDR_p']    = fdr
exhaust_stats_df['sig_label'] = exhaust_stats_df['FDR_p'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
)
exhaust_stats_df.to_csv(table_out / 'fig5e_exhaustion_marker_stats.csv', index=False)
print(exhaust_stats_df.to_string(index=False))

# plot
nrows = len(cell_types)
ncols = len(exhaustion_genes_list)
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4 * ncols, 4 * nrows),
                          sharey=False)
axes = np.atleast_2d(axes)

for r, cell_type in enumerate(cell_types):
    for c, gene in enumerate(exhaustion_genes_list):
        ax = axes[r, c]
        plot_df = exhaust_pct[cell_type][gene].copy()

        sns.boxplot(
            data=plot_df, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, fill=False, showfliers=False, legend=False,
        )
        sns.stripplot(
            data=plot_df, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, edgecolor='k', linewidth=1,
            size=4, alpha=0.7, legend=False,
        )
        if r == 0:
            ax.set_title(gene, fontsize=22, pad=15)
        if c == 0:
            ax.set_ylabel(f'{cell_type}\n% cells expressing', fontsize=18)
        else:
            ax.set_ylabel('')
        ax.set_xlabel('')
        ax.set_xticklabels([])
        ax.spines[['top', 'right']].set_visible(False)

        sig = exhaust_stats_df.loc[
            (exhaust_stats_df['cell_type'] == cell_type) &
            (exhaust_stats_df['gene'] == gene), 'sig_label'
        ].values[0]
        if sig:
            ax.text(0.5, 0.72, sig, ha='center', va='bottom',
                    transform=ax.transAxes, fontsize=48)

handles = [
    plt.Line2D([0], [0], color=palette[g], marker='o', linestyle='',
               markersize=8, markeredgecolor='k', markeredgewidth=1, label=g)
    for g in group_order
]
fig.legend(handles=handles, loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=16)
fig.suptitle('Exhaustion marker expression across lymphocyte cell types by MHC2 cluster',
             fontsize=16, y=1.05)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(fig_out / 'fig5e_exhaustion_markers.pdf', bbox_inches='tight')
plt.show()

In [ ]:
def plot_dual_metric_panel(
    adata,
    genes_dict,
    cell_types,
    group_col='MHC2_clustering',
    group_order=['MHC class II High', 'MHC class II Low'],
    palette={'MHC class II High': '#FF8811FF', 'MHC class II Low': '#462255FF'},
    fig_path=None,
    title=None,
):
    """
    Plot percent expressing and mean expression (among positive cells)
    side by side for each gene × cell type combination.

    Parameters
    ----------
    adata : AnnData
    genes_dict : dict
        {gene_symbol: [ensembl_id, ...]}
    cell_types : list of str
    group_col : str
    group_order : list of str
    palette : dict
    fig_path : Path or None
    title : str or None

    Returns
    -------
    stats_df : pd.DataFrame
    """
    gene_list = list(genes_dict.keys())
    stats_records = []
    pct_data  = {}
    mean_data = {}

    for cell_type in cell_types:
        subset = adata[adata.obs['cell_type_major'] == cell_type]
        pct_data[cell_type]  = {}
        mean_data[cell_type] = {}

        for gene, gene_ens_list in genes_dict.items():
            gene_ens = gene_ens_list[0]
            x = subset.to_df()[gene_ens]
            expr = x.to_frame(name='expr')
            expr['sample']  = subset.obs['sample'].values
            expr['cluster'] = subset.obs[group_col].values

            # % expressing
            pct_df = (
                (expr.assign(detected=(expr['expr'] > 0).astype(int))
                 .groupby(['sample', 'cluster'], observed=True)['detected']
                 .mean() * 100)
                .reset_index()
                .rename(columns={'detected': 'expr'})
            )
            pct_data[cell_type][gene] = pct_df

            # mean among positive cells
            mean_df = (
                expr[expr['expr'] > 0]
                .groupby(['sample', 'cluster'], observed=True)['expr']
                .mean()
                .reset_index()
            )
            mean_data[cell_type][gene] = mean_df

            # stats — run on % expressing
            g1 = pct_df.loc[pct_df['cluster'] == group_order[0], 'expr']
            g2 = pct_df.loc[pct_df['cluster'] == group_order[1], 'expr']
            p  = mannwhitneyu(g1, g2, alternative='two-sided')[1] if len(g1) > 0 and len(g2) > 0 else np.nan
            stats_records.append({'cell_type': cell_type, 'gene': gene, 'p_value': p})

    stats_df = pd.DataFrame(stats_records)
    _, fdr, _, _ = multipletests(stats_df['p_value'].fillna(1), method='fdr_bh')
    stats_df['FDR_p']     = fdr
    stats_df['sig_label'] = stats_df['FDR_p'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    )

    # figure — 2 columns per gene (pct + mean), rows = cell types
    nrows = len(cell_types)
    ncols = len(gene_list) * 2

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3 * ncols, 4 * nrows),
                             sharey=False)
    axes = np.atleast_2d(axes)

    for r, cell_type in enumerate(cell_types):
        for c_gene, gene in enumerate(gene_list):
            for c_metric, (metric_label, data) in enumerate([
                ('% expressing', pct_data[cell_type][gene]),
                ('mean expr\n(pos cells)', mean_data[cell_type][gene]),
            ]):
                col = c_gene * 2 + c_metric
                ax  = axes[r, col]

                sns.boxplot(
                    data=data, x='cluster', y='expr', hue='cluster',
                    order=group_order, palette=palette,
                    ax=ax, fill=False, showfliers=False, legend=False,
                )
                sns.stripplot(
                    data=data, x='cluster', y='expr', hue='cluster',
                    order=group_order, palette=palette,
                    ax=ax, edgecolor='k', linewidth=1,
                    size=3, alpha=0.6, legend=False,
                )

                if r == 0:
                    ax.set_title(f'{gene}\n{metric_label}', fontsize=14, pad=10)
                if c_gene == 0 and c_metric == 0:
                    ax.set_ylabel(cell_type, fontsize=14)
                else:
                    ax.set_ylabel('')

                ax.set_xlabel('')
                ax.set_xticklabels([])
                ax.spines[['top', 'right']].set_visible(False)

                # significance on % expressing column only
                if c_metric == 0:
                    sig = stats_df.loc[
                        (stats_df['cell_type'] == cell_type) &
                        (stats_df['gene'] == gene), 'sig_label'
                    ].values[0]
                    if sig:
                        ax.text(0.5, 0.78, sig, ha='center', va='bottom',
                                transform=ax.transAxes, fontsize=36)

    handles = [
        plt.Line2D([0], [0], color=palette[g], marker='o', linestyle='',
                   markersize=8, markeredgecolor='k', markeredgewidth=1, label=g)
        for g in group_order
    ]
    fig.legend(handles=handles, loc='upper center', ncol=2,
               frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=14)

    if title:
        fig.suptitle(title, fontsize=16, y=1.05)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    if fig_path:
        plt.savefig(fig_path, bbox_inches='tight')

    plt.show()
    return stats_df

In [ ]:
# cytotoxic panel
stats_cytotoxic = plot_dual_metric_panel(
    tumordata,
    genes_dict=genes_dict,
    cell_types=cell_types,
    fig_path=fig_out / 'fig5e_cytotoxic_dual_metric.pdf',
    title='Cytotoxic marker expression across lymphocyte cell types',
)
stats_cytotoxic.to_csv(table_out / 'fig5e_cytotoxic_dual_metric_stats.csv', index=False)

# exhaustion panel
stats_exhaustion = plot_dual_metric_panel(
    tumordata,
    genes_dict=exhaust_dict,
    cell_types=cell_types,
    fig_path=fig_out / 'fig5e_exhaustion_dual_metric.pdf',
    title='Exhaustion marker expression across lymphocyte cell types',
)
stats_exhaustion.to_csv(table_out / 'fig5e_exhaustion_dual_metric_stats.csv', index=False)